Notebook 1 — Acquisition des données SpaceNet 8
Cellule 1 — Installation des bibliothèques

In [15]:
!pip install boto3 pandas scikit-learn tqdm -q

Cellule 2 — Importation des bibliothèques

In [16]:
import os
import re
import json
import random
from pathlib import Path

import boto3
from botocore import UNSIGNED
from botocore.config import Config

import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split

Cellule 3 — Configuration générale

In [17]:
# ==============================
# CONFIGURATION DU NOTEBOOK
# ==============================

SEED = 42
random.seed(SEED)

MAX_PAIRS = 100

BUCKET_NAME = "spacenet-dataset"

# Bon chemin SpaceNet 8 Germany Training Public
BASE_PREFIX = "spacenet/SN8_floods/Germany_Training_Public/"
POST_EVENT_PREFIX = BASE_PREFIX + "POST-event/"
ANNOTATION_PREFIX = BASE_PREFIX + "annotations/"

BASE_DIR = Path("../dattest")
RAW_DIR = BASE_DIR / "rawtest"
IMAGES_DIR = RAW_DIR / "images"
ANNOTATIONS_DIR = RAW_DIR / "annotations"
METADATA_DIR = BASE_DIR / "metadata"

IMAGES_DIR.mkdir(parents=True, exist_ok=True)
ANNOTATIONS_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration terminée.")
print("Post-event prefix :", POST_EVENT_PREFIX)
print("Annotation prefix :", ANNOTATION_PREFIX)
print("Dossier images :", IMAGES_DIR)
print("Dossier annotations :", ANNOTATIONS_DIR)
print("Dossier metadata :", METADATA_DIR)

Configuration terminée.
Post-event prefix : spacenet/SN8_floods/Germany_Training_Public/POST-event/
Annotation prefix : spacenet/SN8_floods/Germany_Training_Public/annotations/
Dossier images : ..\dattest\rawtest\images
Dossier annotations : ..\dattest\rawtest\annotations
Dossier metadata : ..\dattest\metadata


Cellule 4 — Connexion au bucket public SpaceNet

In [18]:
# Connexion anonyme au bucket public SpaceNet
s3 = boto3.client(
    "s3",
    config=Config(signature_version=UNSIGNED)
)

print("Connexion au bucket SpaceNet réussie.")

Connexion au bucket SpaceNet réussie.


Cellule 5 — Fonction pour lister les fichiers S3

In [19]:
def list_s3_files(bucket_name, prefix):
    """
    Liste tous les fichiers disponibles dans un bucket S3 public
    à partir d'un préfixe donné.
    """
    files = []
    continuation_token = None

    while True:
        if continuation_token:
            response = s3.list_objects_v2(
                Bucket=bucket_name,
                Prefix=prefix,
                ContinuationToken=continuation_token
            )
        else:
            response = s3.list_objects_v2(
                Bucket=bucket_name,
                Prefix=prefix
            )

        if "Contents" in response:
            for obj in response["Contents"]:
                files.append(obj["Key"])

        if response.get("IsTruncated"):
            continuation_token = response.get("NextContinuationToken")
        else:
            break

    return files

Cellule 6 — Lister les images et annotations

In [20]:
print("Recherche des images post-event...")
image_files = list_s3_files(BUCKET_NAME, POST_EVENT_PREFIX)

print("Recherche des annotations...")
annotation_files = list_s3_files(BUCKET_NAME, ANNOTATION_PREFIX)

image_files = [
    f for f in image_files
    if f.lower().endswith(".tif")
]

annotation_files = [
    f for f in annotation_files
    if f.lower().endswith(".geojson")
]

print("Images post-event trouvées :", len(image_files))
print("Annotations trouvées :", len(annotation_files))

print("\nExemples images :")
for f in image_files[:5]:
    print(Path(f).name)

print("\nExemples annotations :")
for f in annotation_files[:5]:
    print(Path(f).name)

Recherche des images post-event...
Recherche des annotations...
Images post-event trouvées : 282
Annotations trouvées : 202

Exemples images :
1040050035DC3B00_0_15_63.tif
1040050035DC3B00_0_15_65.tif
1040050035DC3B00_0_15_66.tif
1040050035DC3B00_0_15_67.tif
1040050035DC3B00_0_15_68.tif

Exemples annotations :
0_15_63.geojson
0_15_65.geojson
0_15_66.geojson
0_15_67.geojson
0_15_68.geojson


Cellule 7 — Fonction pour extraire l’identifiant de tuile

In [21]:
def extract_tile_id(path_or_key):
    """
    Extrait l'identifiant de tuile SpaceNet.
    
    Exemples :
    Image      : 1040050035DC3B00_0_15_63.tif -> 0_15_63
    Annotation : 0_15_63.geojson              -> 0_15_63
    """
    name = Path(path_or_key).stem
    
    # Cas annotation : 0_15_63
    if re.fullmatch(r"\d+_\d+_\d+", name):
        return name
    
    # Cas image : 1040050035DC3B00_0_15_63
    match = re.search(r"_(\d+_\d+_\d+)$", name)
    if match:
        return match.group(1)
    
    return None

Cellule 8 — Créer les tables images / annotations

In [22]:
images_df = pd.DataFrame({
    "image_s3_key": image_files
})

images_df["tile_id"] = images_df["image_s3_key"].apply(extract_tile_id)
images_df["image_filename"] = images_df["image_s3_key"].apply(lambda x: Path(x).name)

annotations_df = pd.DataFrame({
    "annotation_s3_key": annotation_files
})

annotations_df["tile_id"] = annotations_df["annotation_s3_key"].apply(extract_tile_id)
annotations_df["annotation_filename"] = annotations_df["annotation_s3_key"].apply(lambda x: Path(x).name)

images_df = images_df.dropna(subset=["tile_id"]).reset_index(drop=True)
annotations_df = annotations_df.dropna(subset=["tile_id"]).reset_index(drop=True)

print("Images avec tile_id :", len(images_df))
print("Annotations avec tile_id :", len(annotations_df))

display(images_df.head())
display(annotations_df.head())

Images avec tile_id : 282
Annotations avec tile_id : 202


,image_s3_key,tile_id,image_filename
0,spacenet/SN8_floods/Germany_Training_Public/PO...,0_15_63,1040050035DC3B00_0_15_63.tif
1,spacenet/SN8_floods/Germany_Training_Public/PO...,0_15_65,1040050035DC3B00_0_15_65.tif
2,spacenet/SN8_floods/Germany_Training_Public/PO...,0_15_66,1040050035DC3B00_0_15_66.tif
3,spacenet/SN8_floods/Germany_Training_Public/PO...,0_15_67,1040050035DC3B00_0_15_67.tif
4,spacenet/SN8_floods/Germany_Training_Public/PO...,0_15_68,1040050035DC3B00_0_15_68.tif


,annotation_s3_key,tile_id,annotation_filename
0,spacenet/SN8_floods/Germany_Training_Public/an...,0_15_63,0_15_63.geojson
1,spacenet/SN8_floods/Germany_Training_Public/an...,0_15_65,0_15_65.geojson
2,spacenet/SN8_floods/Germany_Training_Public/an...,0_15_66,0_15_66.geojson
3,spacenet/SN8_floods/Germany_Training_Public/an...,0_15_67,0_15_67.geojson
4,spacenet/SN8_floods/Germany_Training_Public/an...,0_15_68,0_15_68.geojson


Cellule 9 — Appariement image / annotation

In [23]:
pairs_df = pd.merge(
    images_df,
    annotations_df,
    on="tile_id",
    how="inner"
)

pairs_df = pairs_df.drop_duplicates(subset=["tile_id"]).reset_index(drop=True)

print("Nombre de paires image/annotation trouvées :", len(pairs_df))

display(pairs_df.head())

Nombre de paires image/annotation trouvées : 202


,image_s3_key,tile_id,image_filename,annotation_s3_key,annotation_filename
0,spacenet/SN8_floods/Germany_Training_Public/PO...,0_15_63,1040050035DC3B00_0_15_63.tif,spacenet/SN8_floods/Germany_Training_Public/an...,0_15_63.geojson
1,spacenet/SN8_floods/Germany_Training_Public/PO...,0_15_65,1040050035DC3B00_0_15_65.tif,spacenet/SN8_floods/Germany_Training_Public/an...,0_15_65.geojson
2,spacenet/SN8_floods/Germany_Training_Public/PO...,0_15_66,1040050035DC3B00_0_15_66.tif,spacenet/SN8_floods/Germany_Training_Public/an...,0_15_66.geojson
3,spacenet/SN8_floods/Germany_Training_Public/PO...,0_15_67,1040050035DC3B00_0_15_67.tif,spacenet/SN8_floods/Germany_Training_Public/an...,0_15_67.geojson
4,spacenet/SN8_floods/Germany_Training_Public/PO...,0_15_68,1040050035DC3B00_0_15_68.tif,spacenet/SN8_floods/Germany_Training_Public/an...,0_15_68.geojson


Cellule 10 — Sélection aléatoire des paires

In [24]:
if len(pairs_df) == 0:
    raise ValueError("Aucune paire image/annotation trouvée. Vérifie le préfixe S3.")

# Si MAX_PAIRS dépasse le nombre disponible, on prend tout
n_pairs = min(MAX_PAIRS, len(pairs_df))

selected_df = pairs_df.sample(
    n=n_pairs,
    random_state=SEED
).reset_index(drop=True)

print("Nombre de paires sélectionnées :", len(selected_df))

display(selected_df.head())

Nombre de paires sélectionnées : 100


,image_s3_key,tile_id,image_filename,annotation_s3_key,annotation_filename
0,spacenet/SN8_floods/Germany_Training_Public/PO...,0_30_59,10500500E6DD3C00_0_30_59.tif,spacenet/SN8_floods/Germany_Training_Public/an...,0_30_59.geojson
1,spacenet/SN8_floods/Germany_Training_Public/PO...,0_17_68,1040050035DC3B00_0_17_68.tif,spacenet/SN8_floods/Germany_Training_Public/an...,0_17_68.geojson
2,spacenet/SN8_floods/Germany_Training_Public/PO...,0_20_67,1040050035DC3B00_0_20_67.tif,spacenet/SN8_floods/Germany_Training_Public/an...,0_20_67.geojson
3,spacenet/SN8_floods/Germany_Training_Public/PO...,0_39_61,10500500E6DD3C00_0_39_61.tif,spacenet/SN8_floods/Germany_Training_Public/an...,0_39_61.geojson
4,spacenet/SN8_floods/Germany_Training_Public/PO...,0_42_62,10500500E6DD3C00_0_42_62.tif,spacenet/SN8_floods/Germany_Training_Public/an...,0_42_62.geojson


Cellule 11 — Création du split train / validation / test

In [25]:
# Split recommandé :
# 70% train, 15% validation, 15% test

train_df, temp_df = train_test_split(
    selected_df,
    test_size=0.30,
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED
)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

metadata_df = pd.concat(
    [train_df, val_df, test_df],
    axis=0
).reset_index(drop=True)

print("Répartition des données :")
print(metadata_df["split"].value_counts())

display(metadata_df.head())

Répartition des données :
split
train    70
val      15
test     15
Name: count, dtype: int64


,image_s3_key,tile_id,image_filename,annotation_s3_key,annotation_filename,split
0,spacenet/SN8_floods/Germany_Training_Public/PO...,0_37_68,10500500E6DD3C00_0_37_68.tif,spacenet/SN8_floods/Germany_Training_Public/an...,0_37_68.geojson,train
1,spacenet/SN8_floods/Germany_Training_Public/PO...,0_18_67,1040050035DC3B00_0_18_67.tif,spacenet/SN8_floods/Germany_Training_Public/an...,0_18_67.geojson,train
2,spacenet/SN8_floods/Germany_Training_Public/PO...,0_29_61,10500500E6DD3C00_0_29_61.tif,spacenet/SN8_floods/Germany_Training_Public/an...,0_29_61.geojson,train
3,spacenet/SN8_floods/Germany_Training_Public/PO...,0_32_69,10500500E6DD3C00_0_32_69.tif,spacenet/SN8_floods/Germany_Training_Public/an...,0_32_69.geojson,train
4,spacenet/SN8_floods/Germany_Training_Public/PO...,0_21_65,1040050035DC3B00_0_21_65.tif,spacenet/SN8_floods/Germany_Training_Public/an...,0_21_65.geojson,train


Cellule 12 — Ajouter les chemins locaux

In [26]:
metadata_df["local_image_path"] = metadata_df["image_filename"].apply(
    lambda x: str(IMAGES_DIR / x)
)

metadata_df["local_annotation_path"] = metadata_df["annotation_filename"].apply(
    lambda x: str(ANNOTATIONS_DIR / x)
)

display(metadata_df.head())

,image_s3_key,tile_id,image_filename,annotation_s3_key,annotation_filename,split,local_image_path,local_annotation_path
0,spacenet/SN8_floods/Germany_Training_Public/PO...,0_37_68,10500500E6DD3C00_0_37_68.tif,spacenet/SN8_floods/Germany_Training_Public/an...,0_37_68.geojson,train,..\dattest\rawtest\images\10500500E6DD3C00_0_3...,..\dattest\rawtest\annotations\0_37_68.geojson
1,spacenet/SN8_floods/Germany_Training_Public/PO...,0_18_67,1040050035DC3B00_0_18_67.tif,spacenet/SN8_floods/Germany_Training_Public/an...,0_18_67.geojson,train,..\dattest\rawtest\images\1040050035DC3B00_0_1...,..\dattest\rawtest\annotations\0_18_67.geojson
2,spacenet/SN8_floods/Germany_Training_Public/PO...,0_29_61,10500500E6DD3C00_0_29_61.tif,spacenet/SN8_floods/Germany_Training_Public/an...,0_29_61.geojson,train,..\dattest\rawtest\images\10500500E6DD3C00_0_2...,..\dattest\rawtest\annotations\0_29_61.geojson
3,spacenet/SN8_floods/Germany_Training_Public/PO...,0_32_69,10500500E6DD3C00_0_32_69.tif,spacenet/SN8_floods/Germany_Training_Public/an...,0_32_69.geojson,train,..\dattest\rawtest\images\10500500E6DD3C00_0_3...,..\dattest\rawtest\annotations\0_32_69.geojson
4,spacenet/SN8_floods/Germany_Training_Public/PO...,0_21_65,1040050035DC3B00_0_21_65.tif,spacenet/SN8_floods/Germany_Training_Public/an...,0_21_65.geojson,train,..\dattest\rawtest\images\1040050035DC3B00_0_2...,..\dattest\rawtest\annotations\0_21_65.geojson


Cellule 13 — Fonction de téléchargement

In [27]:
def download_s3_file(bucket_name, s3_key, local_path):
    """
    Télécharge un fichier S3 vers un chemin local.
    Si le fichier existe déjà, il n'est pas retéléchargé.
    """
    local_path = Path(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)

    if local_path.exists() and local_path.stat().st_size > 0:
        return "exists"

    s3.download_file(bucket_name, s3_key, str(local_path))
    return "downloaded"

Cellule 14 — Télécharger les images et annotations

In [28]:
download_logs = []

print("Téléchargement des images et annotations...")

for _, row in tqdm(metadata_df.iterrows(), total=len(metadata_df)):
    
    # Télécharger image
    image_status = download_s3_file(
        BUCKET_NAME,
        row["image_s3_key"],
        row["local_image_path"]
    )
    
    # Télécharger annotation
    annotation_status = download_s3_file(
        BUCKET_NAME,
        row["annotation_s3_key"],
        row["local_annotation_path"]
    )
    
    download_logs.append({
        "tile_id": row["tile_id"],
        "image_filename": row["image_filename"],
        "annotation_filename": row["annotation_filename"],
        "image_status": image_status,
        "annotation_status": annotation_status,
        "split": row["split"]
    })

download_logs_df = pd.DataFrame(download_logs)

print("Téléchargement terminé.")
display(download_logs_df.head())

Téléchargement des images et annotations...


100%|██████████| 100/100 [10:53<00:00,  6.54s/it]

Téléchargement terminé.


,tile_id,image_filename,annotation_filename,image_status,annotation_status,split
0,0_37_68,10500500E6DD3C00_0_37_68.tif,0_37_68.geojson,downloaded,downloaded,train
1,0_18_67,1040050035DC3B00_0_18_67.tif,0_18_67.geojson,exists,exists,train
2,0_29_61,10500500E6DD3C00_0_29_61.tif,0_29_61.geojson,downloaded,downloaded,train
3,0_32_69,10500500E6DD3C00_0_32_69.tif,0_32_69.geojson,downloaded,downloaded,train
4,0_21_65,1040050035DC3B00_0_21_65.tif,0_21_65.geojson,downloaded,downloaded,train


Cellule 15 — Vérification des fichiers téléchargés

In [29]:
metadata_df["image_exists"] = metadata_df["local_image_path"].apply(
    lambda x: Path(x).exists() and Path(x).stat().st_size > 0
)

metadata_df["annotation_exists"] = metadata_df["local_annotation_path"].apply(
    lambda x: Path(x).exists() and Path(x).stat().st_size > 0
)

print("Images téléchargées :", metadata_df["image_exists"].sum(), "/", len(metadata_df))
print("Annotations téléchargées :", metadata_df["annotation_exists"].sum(), "/", len(metadata_df))

missing_files = metadata_df[
    (~metadata_df["image_exists"]) | (~metadata_df["annotation_exists"])
]

if len(missing_files) > 0:
    print("Attention : certains fichiers sont manquants.")
    display(missing_files)
else:
    print("Tous les fichiers sont bien téléchargés.")

Images téléchargées : 100 / 100
Annotations téléchargées : 100 / 100
Tous les fichiers sont bien téléchargés.


Cellule 16 — Sauvegarder les métadonnées

In [30]:
metadata_csv_path = METADATA_DIR / "metadata_pairs.csv"
download_log_path = METADATA_DIR / "download_logs.csv"

metadata_df.to_csv(metadata_csv_path, index=False)
download_logs_df.to_csv(download_log_path, index=False)

print("Fichier metadata sauvegardé :", metadata_csv_path)
print("Fichier logs sauvegardé :", download_log_path)

Fichier metadata sauvegardé : ..\dattest\metadata\metadata_pairs.csv
Fichier logs sauvegardé : ..\dattest\metadata\download_logs.csv


Cellule 17 — Sauvegarder les splits en JSON

In [31]:
splits = {
    "train": metadata_df[metadata_df["split"] == "train"]["tile_id"].tolist(),
    "val": metadata_df[metadata_df["split"] == "val"]["tile_id"].tolist(),
    "test": metadata_df[metadata_df["split"] == "test"]["tile_id"].tolist()
}

splits_path = METADATA_DIR / "splits.json"

with open(splits_path, "w", encoding="utf-8") as f:
    json.dump(splits, f, indent=4, ensure_ascii=False)

print("Fichier splits sauvegardé :", splits_path)

print("\nNombre train :", len(splits["train"]))
print("Nombre val :", len(splits["val"]))
print("Nombre test :", len(splits["test"]))

Fichier splits sauvegardé : ..\dattest\metadata\splits.json

Nombre train : 70
Nombre val : 15
Nombre test : 15


Cellule 18 — Résumé final du notebook

In [32]:
print("=" * 60)
print("RÉSUMÉ ACQUISITION")
print("=" * 60)

print("Dataset utilisé : SpaceNet 8 Floods - Germany")
print("Nombre total de paires disponibles :", len(pairs_df))
print("Nombre de paires sélectionnées :", len(metadata_df))

print("\nRépartition :")
print(metadata_df["split"].value_counts())

print("\nDossiers créés :")
print("Images :", IMAGES_DIR)
print("Annotations :", ANNOTATIONS_DIR)
print("Metadata :", METADATA_DIR)

print("\nFichiers générés :")
print("- metadata_pairs.csv")
print("- download_logs.csv")
print("- splits.json")

print("\nAcquisition terminée avec succès.")

RÉSUMÉ ACQUISITION
Dataset utilisé : SpaceNet 8 Floods - Germany
Nombre total de paires disponibles : 202
Nombre de paires sélectionnées : 100

Répartition :
split
train    70
val      15
test     15
Name: count, dtype: int64

Dossiers créés :
Images : ..\dattest\rawtest\images
Annotations : ..\dattest\rawtest\annotations
Metadata : ..\dattest\metadata

Fichiers générés :
- metadata_pairs.csv
- download_logs.csv
- splits.json

Acquisition terminée avec succès.
